In [ ]:
# Cell 1
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when
from pyspark.ml.feature import Tokenizer, StopWordsRemover, HashingTF, IDF
from pyspark.ml.classification import LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator

spark = SparkSession.builder \
    .appName("SentimentTraining") \
    .config("spark.sql.adaptive.enabled", "true") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print("Spark ready!")

In [ ]:
# Cell 2
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("../data/processed/yelp_sample_csv/part-00000-5d7d137c-d451-4b65-b9c2-3c3d0bbcd7f8-c000.csv")

print(f"Loaded {df.count()} reviews")
df.printSchema()

In [ ]:
# Cell 3
df_labeled = df.filter(col("review_stars").isin([1, 2, 4, 5])) \
    .filter(col("clean_text").isNotNull()) \
    .withColumn("label", when(col("review_stars") >= 4, 1.0).otherwise(0.0))

print(f"Labeled reviews: {df_labeled.count()}")
print(f"Positive: {df_labeled.filter(col('label')==1).count()}")
print(f"Negative: {df_labeled.filter(col('label')==0).count()}")

In [ ]:
# Cell 4
train_data, test_data = df_labeled.randomSplit([0.8, 0.2], seed=42)
print(f"Train: {train_data.count()}, Test: {test_data.count()}")

In [ ]:
# Cell 5
tokenizer = Tokenizer(inputCol="text_clean", outputCol="words")
remover = StopWordsRemover(inputCol="words", outputCol="filtered_words")
hashingTF = HashingTF(inputCol="filtered_words", outputCol="raw_features", numFeatures=5000)
idf = IDF(inputCol="raw_features", outputCol="features")
lr = LogisticRegression(featuresCol="features", labelCol="label",predictionCol="prediction", probabilityCol="probability",maxIter=30, regParam=0.01)

pipeline = Pipeline(stages=[tokenizer, remover, hashingTF, idf, lr])
print("Pipeline built")

In [ ]:
# Cell 6
print("Training model...")
model = pipeline.fit(train_data)
print("Training complete!")

In [ ]:
# Cell 7
predictions = model.transform(test_data)

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction",metricName="areaUnderROC")
auc = evaluator.evaluate(predictions)

correct = predictions.filter(col("label") == col("prediction")).count()
total = predictions.count()
accuracy = correct / total

print(f"Model Performance:")
print(f"Accuracy: {accuracy:.2%}")
print(f"AUC Score: {auc:.2%}")

In [ ]:
# Cell 8
def get_rating_label(prob_pos):
    if prob_pos >= 90:
        return "EXCELLENT"
    elif prob_pos >= 75:
        return "GOOD"
    elif prob_pos >= 50:
        return "NOT BAD"
    elif prob_pos >= 25:
        return "BAD"
    else:
        return "TERRIBLE"

test_reviews = [
    "The food was absolutely amazing! Best meal ever.",
    "Terrible service, rude staff, cold food.",
    "Pretty good experience, would come again.",
    "It was okay, nothing special.",
    "I love this place! Highly recommend!"
]

print("\n" + "="*60)
print("SENTIMENT PREDICTIONS (with probability)")
print("="*60)

for review in test_reviews:
    
    df_test = spark.createDataFrame([(review,)], ["text_clean"])
    
    result = model.transform(df_test)
    
    prob = result.select("probability").collect()[0][0][1] * 100
    
    rating = get_rating_label(prob)
    
    print(f"\nReview: {review}")
    print(f"{prob:.1f}% positive, {100-prob:.1f}% negative")
    print(f"Rating: {rating}")

In [ ]:
# Cell 9
model_path = "/tmp/sentiment_model"
model.write().overwrite().save(model_path)
print(f"Model saved to {model_path}")

In [ ]:
# Cell 10
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score

os.makedirs("output/plots", exist_ok=True)
os.makedirs("output/metrics", exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("Set2")

print("Visualization libraries imported!")

In [ ]:
# Cell 11
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

pred_pd = predictions.select("label", "prediction").toPandas()
y_true = pred_pd['label'].values
y_pred = pred_pd['prediction'].values

y_scores = predictions.select("probability").rdd.map(lambda x: x[0][1]).collect()

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
auc = roc_auc_score(y_true, y_scores)

correct = predictions.filter(col("label") == col("prediction")).count()
total = predictions.count()
spark_accuracy = correct / total

print("="*60)
print("MODEL PERFORMANCE METRICS")
print("="*60)
print(f"Accuracy:  {accuracy:.2%} (Spark: {spark_accuracy:.2%})")
print(f"Precision: {precision:.2%}")
print(f"Recall:    {recall:.2%}")
print(f"F1 Score:  {f1:.2%}")
print(f"AUC Score: {auc:.2%}")
print("="*60)

metrics = {
    "accuracy": round(accuracy * 100, 2),
    "precision": round(precision * 100, 2),
    "recall": round(recall * 100, 2),
    "f1_score": round(f1 * 100, 2),
    "auc": round(auc * 100, 2)
}

with open("output/metrics/sentiment_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"\n✅ Metrics saved to: output/metrics/sentiment_metrics.json")

In [ ]:
# Cell 12
cm = confusion_matrix(y_true, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'],
            ax=ax, annot_kws={'size': 14})

ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
ax.set_title('Confusion Matrix - Sentiment Analysis', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('output/plots/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: output/plots/confusion_matrix.png")

In [ ]:
# Cell 13
fig, ax = plt.subplots(figsize=(8, 5))

metrics_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC']
metrics_values = [accuracy * 100, precision * 100, recall * 100, f1 * 100, auc * 100]
colors = ['#2E86AB', '#A23B72', '#F18F01', '#048A81', '#6A4C93']

bars = ax.bar(metrics_names, metrics_values, color=colors, alpha=0.8, edgecolor='black')

for bar, value in zip(bars, metrics_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{value:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.set_ylim(0, 105)
ax.set_ylabel('Percentage (%)', fontsize=12)
ax.set_title('Sentiment Model Performance Metrics', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('output/plots/model_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: output/plots/model_performance.png")

In [ ]:
# Cell 14
fpr, tpr, thresholds = roc_curve(y_true, y_scores)

fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC Curve (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.3, color='orange')

ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve - Sentiment Classification', fontsize=14, fontweight='bold')
ax.legend(loc="lower right")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('output/plots/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: output/plots/roc_curve.png")

In [ ]:
# Cell 15

preds = predictions.select("prediction").rdd.map(lambda x: x[0]).collect()
probs = predictions.select("probability").rdd.map(lambda x: x[0][1]).collect()

prob_df = pd.DataFrame({
    'probability_positive': probs,
    'true_label': y_true,
    'prediction': preds,
    'correct': y_true == preds
})

correct_pos = prob_df[(prob_df['correct'] == True) & (prob_df['true_label'] == 1)]['probability_positive']
correct_neg = prob_df[(prob_df['correct'] == True) & (prob_df['true_label'] == 0)]['probability_positive']
incorrect = prob_df[prob_df['correct'] == False]['probability_positive']

fig, ax = plt.subplots(figsize=(10, 6))

if len(correct_pos) > 0:
    ax.hist(correct_pos, bins=20, alpha=0.7, label='Correct - Positive Reviews', color='green', edgecolor='black')
if len(correct_neg) > 0:
    ax.hist(correct_neg, bins=20, alpha=0.7, label='Correct - Negative Reviews', color='blue', edgecolor='black')
if len(incorrect) > 0:
    ax.hist(incorrect, bins=20, alpha=0.7, label='Incorrect Predictions', color='red', edgecolor='black')

ax.set_xlabel('Probability of Positive Class', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Prediction Probability Distribution', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('output/plots/probability_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: output/plots/probability_distribution.png")

In [ ]:
# Cell 16
pos_probs = [p for p, l in zip(probs, y_true) if l == 1]
neg_probs = [p for p, l in zip(probs, y_true) if l == 0]

fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(pos_probs, bins=20, alpha=0.7, label='True Positive Reviews', color='green', edgecolor='black')
ax.hist(neg_probs, bins=20, alpha=0.7, label='True Negative Reviews', color='red', edgecolor='black')

ax.set_xlabel('Predicted Positive Probability', fontsize=12)
ax.set_ylabel('Frequency', fontsize=12)
ax.set_title('Sentiment Score Distribution by True Label', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('output/plots/sentiment_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: output/plots/sentiment_distribution.png")

In [ ]:
# Cell 17
def get_rating_label(prob_pos):
    if prob_pos >= 90:
        return "EXCELLENT"
    elif prob_pos >= 75:
        return "GOOD"
    elif prob_pos >= 50:
        return "NOT BAD"
    elif prob_pos >= 25:
        return "BAD"
    else:
        return "TERRIBLE"

test_reviews = [
    "The food was absolutely amazing! Best meal ever.",
    "Terrible service, rude staff, cold food.",
    "Pretty good experience, would come again.",
    "It was okay, nothing special.",
    "I love this place! Highly recommend!",
    "Worst restaurant ever! Never coming back.",
    "Great atmosphere and friendly staff."
]

review_short_labels = ['Amazing Meal', 'Terrible Service', 'Good Experience', 
                       'Okay Nothing Special', 'Love It!', 'Worst Ever', 'Great Atmosphere']

test_probs = []
test_ratings = []

for review in test_reviews:
    df_test = spark.createDataFrame([(review,)], ["clean_text"])
    result = model.transform(df_test)
    prob = result.select("probability").collect()[0][0][1] * 100
    test_probs.append(prob)
    test_ratings.append(get_rating_label(prob))

results_df = pd.DataFrame({
    'Review': review_short_labels,
    'Probability (%)': [f"{p:.1f}%" for p in test_probs],
    'Rating': test_ratings
})

print("\n" + "="*60)
print("EXAMPLE REVIEW PREDICTIONS")
print("="*60)
print(results_df.to_string(index=False))
print("="*60)

fig, ax = plt.subplots(figsize=(10, 6))
colors_probs = ['green' if p >= 50 else 'red' for p in test_probs]
bars = ax.barh(review_short_labels, test_probs, color=colors_probs, alpha=0.8, edgecolor='black')

for bar, prob in zip(bars, test_probs):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'{prob:.1f}%', va='center', fontsize=10, fontweight='bold')
    
    rating = get_rating_label(prob)
    ax.text(bar.get_width() - 5, bar.get_y() + bar.get_height()/2,
            f'[{rating}]', va='center', ha='right', fontsize=9, fontweight='bold', color='white')

ax.set_xlim(0, 105)
ax.set_xlabel('Positive Sentiment Probability (%)', fontsize=12)
ax.set_title('Sentiment Predictions for Example Reviews', fontsize=14, fontweight='bold')
ax.axvline(x=50, color='gray', linestyle='--', linewidth=2, label='Neutral Threshold (50%)')
ax.legend()
ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig('output/plots/example_predictions.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: output/plots/example_predictions.png")

In [ ]:
# Cell 18
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Sentiment Model Summary Dashboard', fontsize=16, fontweight='bold')

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'], 
            yticklabels=['Negative', 'Positive'], 
            ax=axes[0,0], annot_kws={'size': 14})
axes[0,0].set_title('Confusion Matrix', fontweight='bold')

axes[0,1].bar(metrics_names, metrics_values, color=colors)
axes[0,1].set_ylim(0, 105)
axes[0,1].set_ylabel('Percentage (%)')
axes[0,1].set_title('Performance Metrics', fontweight='bold')
for i, v in enumerate(metrics_values):
    axes[0,1].text(i, v + 2, f'{v:.1f}%', ha='center', fontsize=9)

axes[1,0].plot(fpr, tpr, 'darkorange', lw=2, label=f'AUC = {auc:.3f}')
axes[1,0].plot([0, 1], [0, 1], 'navy', linestyle='--')
axes[1,0].set_xlabel('False Positive Rate')
axes[1,0].set_ylabel('True Positive Rate')
axes[1,0].set_title('ROC Curve', fontweight='bold')
axes[1,0].legend()
axes[1,0].grid(alpha=0.3)

top5_reviews = review_short_labels[:5]
top5_probs = test_probs[:5]
top5_colors = ['green' if p >= 50 else 'red' for p in top5_probs]
axes[1,1].barh(top5_reviews, top5_probs, color=top5_colors)
axes[1,1].set_xlim(0, 100)
axes[1,1].set_xlabel('Positive Probability (%)')
axes[1,1].set_title('Example Predictions', fontweight='bold')
axes[1,1].axvline(x=50, color='gray', linestyle='--')

plt.tight_layout()
plt.savefig('output/plots/summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Saved: output/plots/summary_dashboard.png")

In [ ]:
# Cell 19
spark.stop()
print("Spark session stopped")